# SFT-разогрев T-lite с LoRA

Решение 03.11.2026 (`docs/decisions.md`, раздел 9). Примеры собираются в токены функцией среды, как роллауты RL;
loss — только по репликам модели; `transformers.Trainer` + LoRA (peft), логиты — только в позициях с меткой.
Отложено 9 тем SFT: по loss на них выбирается эпоха. Результат — `runs/sft/<дата>/` (адаптер, журнал, разбиение).

Перед запуском: `pip install --user -e ".[models,training]"`; веса T-lite должны быть доступны (Hugging Face или
локальный кеш в `HF_HUB_CACHE`).

In [ ]:
import json, os, sys, time
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # перед запуском проверить nvidia-smi
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
import torch
from presentation_rl.environment import build_prompt
from presentation_rl.model_loader import load_model
from presentation_rl.prompts import system_prompt
from presentation_rl.training import sft

config = sft.SFTConfig.load(ROOT / "configs" / "experiments" / "sft_tlite.yaml")
OUT = ROOT / "runs" / "sft" / time.strftime("%Y%m%d_%H%M")
OUT.mkdir(parents=True, exist_ok=True)
print(config, OUT, torch.cuda.get_device_name(0), sep="\n")

In [ ]:
start = time.time()
model, tokenizer = load_model(config.model_id, config.revision, dtype="float16", device_map={"": 0})
print(f"загрузка {time.time() - start:.0f} с, память {torch.cuda.memory_allocated() / 2**30:.1f} ГБ")

## Примеры токенами

Эпизод каждого примера собирается так же, как роллаут: начало — шаблон чата (системный промпт, инструменты, запрос),
дальше реплики модели и выдача инструментов в порядке среды. Слишком длинные эпизоды не обрезаются (оборвался бы
ответ), а отбрасываются — их список печатается.

In [ ]:
def prompt(request):
    return build_prompt(tokenizer, system_prompt(request), request)


samples = sft.build_samples(ROOT, tokenizer, prompt)
samples, dropped = sft.drop_long(samples, config.max_tokens)
print("отброшены как слишком длинные:", [(s.id, s.kind, len(s.ids)) for s in dropped])
train_samples, eval_samples = sft.split(samples, config.eval_tasks, config.seed)
for name, part in (("обучение", train_samples), ("отложенные", eval_samples)):
    print(name, sft.lengths(part))
(OUT / "split.json").write_text(json.dumps({"eval_topics": sorted({s.id for s in eval_samples}),
                                            "dropped": [[s.id, s.kind, len(s.ids)] for s in dropped]},
                                           ensure_ascii=False, indent=1), encoding="utf-8")

## Обучение

In [ ]:
start = time.time()
torch.cuda.reset_peak_memory_stats()
model = sft.train(config, train_samples, eval_samples, OUT, model=model, tokenizer=tokenizer)
print(f"обучение {(time.time() - start) / 60:.0f} мин, пик памяти {torch.cuda.max_memory_allocated() / 2**30:.1f} ГБ")
log = json.loads((OUT / "log.json").read_text(encoding="utf-8"))
for row in log:
    if "eval_loss" in row:
        print(f"эпоха {row['epoch']:.0f}: loss на отложенных темах {row['eval_loss']:.4f}")

## Быстрая проверка

Один ответ на отложенную тему без поиска (полная проверка на validation — EXP-015, отдельная тетрадка).

In [ ]:
from presentation_rl.dsl import parse
from presentation_rl.environment import run_episode
from presentation_rl.model_loader import hf_generator

model.eval()
model.config.use_cache = True
record = next(json.loads(l) for l in (ROOT / sft.PLAIN).read_text(encoding="utf-8").splitlines()
              if json.loads(l)["id"] in {s.id for s in eval_samples})
from presentation_rl.tasks import load_tasks
request = sft.request_for(record, {t.id: t for t in load_tasks(ROOT / "data" / "tasks" / "tasks.json", split="sft")})
episode = run_episode(prompt(request), tokenizer, hf_generator(model, tokenizer, temperature=0.9), None, max_new_tokens=6000)
deck = parse(episode.answer)
print(f"{record['id']}: слайдов {len(deck.slides)}, корректных строк {deck.parse_rate:.0%}, конец — {episode.finish}")
print(episode.answer[:3000])